## Лабораторная работа 8: Фильтрация и свертка

### Упражнение 8.1: Преобразование Фурье гауссовой кривой

In [ ]:
import sys
sys.path.insert(0, '../ThinkDSP/code')

from thinkdsp import decorate
import matplotlib.pyplot as plt
import numpy as np
import scipy.signal

### Свойство гауссовой функции

Преобразование Фурье гауссовой кривой также является гауссовой кривой.

In [ ]:
# Создание гауссовой функции
gaussian = scipy.signal.gaussian(M=32, std=2)
gaussian /= sum(gaussian)  # Нормализация

plt.plot(gaussian)
plt.title('Гауссова функция (std=2)')
decorate(xlabel='Индекс', ylabel='Амплитуда')
plt.show()

Вычислим FFT:

In [ ]:
fft_gaussian = np.fft.fft(gaussian)

plt.plot(abs(fft_gaussian))
plt.title('FFT гауссовой функции')
decorate(xlabel='Частота', ylabel='Амплитуда')
plt.show()

Сдвинем отрицательные частоты влево для лучшей визуализации:

In [ ]:
# Используем fftshift для центрирования
fft_shifted = np.fft.fftshift(fft_gaussian)

plt.plot(abs(fft_shifted))
plt.title('FFT гауссовой функции (центрированный)')
decorate(xlabel='Частота', ylabel='Амплитуда')
plt.show()

**Комментарий:** Видно, что FFT также имеет гауссову форму, что подтверждает теоретическое свойство.

### Влияние параметра std

Исследуем, как изменение стандартного отклонения влияет на FFT.

In [ ]:
stds = [1, 2, 4, 8]
fig, axes = plt.subplots(len(stds), 2, figsize=(12, 10))

for i, std in enumerate(stds):
    # Создание гауссовой функции
    gaussian = scipy.signal.gaussian(M=64, std=std)
    gaussian /= sum(gaussian)
    
    # Исходная функция
    axes[i, 0].plot(gaussian)
    axes[i, 0].set_title(f'Гауссова функция (std={std})')
    axes[i, 0].set_xlabel('Индекс')
    axes[i, 0].set_ylabel('Амплитуда')
    axes[i, 0].grid(True, alpha=0.3)
    
    # FFT
    fft_gaussian = np.fft.fft(gaussian)
    fft_shifted = np.fft.fftshift(fft_gaussian)
    
    axes[i, 1].plot(abs(fft_shifted))
    axes[i, 1].set_title(f'FFT (std={std})')
    axes[i, 1].set_xlabel('Частота')
    axes[i, 1].set_ylabel('Амплитуда')
    axes[i, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Комментарий:** При увеличении std исходная гауссова функция становится шире, а её FFT - уже. Это демонстрирует обратную зависимость между шириной функции во временной и частотной областях.

### Количественный анализ

In [ ]:
# Измерим ширину на половине высоты (FWHM)
def measure_width(signal):
    """
    Измеряет ширину на половине максимума
    """
    max_val = np.max(signal)
    half_max = max_val / 2
    
    # Находим точки, где сигнал превышает половину максимума
    above_half = signal > half_max
    indices = np.where(above_half)[0]
    
    if len(indices) > 0:
        return indices[-1] - indices[0]
    return 0

stds = np.arange(1, 10, 0.5)
time_widths = []
freq_widths = []

for std in stds:
    gaussian = scipy.signal.gaussian(M=128, std=std)
    gaussian /= sum(gaussian)
    
    # Ширина во временной области
    time_width = measure_width(gaussian)
    time_widths.append(time_width)
    
    # Ширина в частотной области
    fft_gaussian = np.fft.fft(gaussian)
    fft_shifted = np.fft.fftshift(abs(fft_gaussian))
    freq_width = measure_width(fft_shifted)
    freq_widths.append(freq_width)

# Визуализация
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(stds, time_widths, 'o-')
ax1.set_xlabel('Стандартное отклонение')
ax1.set_ylabel('Ширина во временной области')
ax1.set_title('Зависимость ширины от std')
ax1.grid(True, alpha=0.3)

ax2.plot(stds, freq_widths, 'o-', color='orange')
ax2.set_xlabel('Стандартное отклонение')
ax2.set_ylabel('Ширина в частотной области')
ax2.set_title('Ширина FFT')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Проверка обратной зависимости
products = np.array(time_widths) * np.array(freq_widths)
print(f'\nПроизведение ширин (должно быть примерно постоянным):')
print(f'Среднее: {np.mean(products):.2f}')
print(f'Стандартное отклонение: {np.std(products):.2f}')

**Комментарий:** Произведение ширин в временной и частотной областях остается примерно постоянным, что соответствует принципу неопределенности Фурье.